# KIS · SigLIP2-384 embedding cho 161k keyframe

Input: Kaggle Dataset chứa bundle từ `tools/prepare_siglip2_bundle.py`
(`manifest.json`, `mapping/<video>.csv`, `images/<video>/<n>.jpg`, ảnh đã là 384×384).

Output `/kaggle/working/siglip2-features.zip`:
`siglip2-features/<group>/<video>/<video>.npy` (float16, L2-normalized, `rows × 768`,
đúng thứ tự dòng mapping CSV) + `SIGLIP2_MANIFEST.json`.

Settings: **GPU T4 x1** (hoặc P100), **Internet ON** (tải model lần đầu).
Ảnh thiếu được ghi `NaN` và liệt kê trong manifest, không tự tạo vector giả.

In [ ]:
import subprocess, sys
try:
    import open_clip
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.31"])
    import open_clip
import os, json, time, hashlib, zipfile
from pathlib import Path
import numpy as np, torch
from PIL import Image
print("open_clip", open_clip.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
MODEL_REPO = "timm/ViT-B-16-SigLIP2-384"
MODEL_REVISION = "369333c629e1ab4eb90806c8e13df3054c3c1e63"  # pinned; đổi = phải embed lại
BATCH_SIZE = int(os.environ.get("KIS_BATCH", 256))
NUM_WORKERS = int(os.environ.get("KIS_WORKERS", 4))
LIMIT_VIDEOS = int(os.environ.get("KIS_LIMIT_VIDEOS", 0))  # 0 = tất cả
OUTPUT_DIR = Path(os.environ.get("KIS_OUTPUT", "/kaggle/working"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def find_bundle():
    if os.environ.get("KIS_BUNDLE"):
        return Path(os.environ["KIS_BUNDLE"])
    hits = sorted(Path("/kaggle/input").rglob("manifest.json"))
    hits = [p.parent for p in hits if (p.parent / "images").is_dir() and (p.parent / "mapping").is_dir()]
    if len(hits) != 1:
        raise SystemExit(f"Cần đúng 1 bundle trong /kaggle/input, thấy {len(hits)}: {hits}")
    return hits[0]

BUNDLE = find_bundle()
manifest = json.loads((BUNDLE / "manifest.json").read_text())
videos = sorted(manifest["videos"])[: LIMIT_VIDEOS or None]
print(BUNDLE, "|", len(videos), "videos |", sum(manifest["videos"][v]["rows"] for v in videos), "rows |",
      manifest.get("missing"), "missing in bundle")

In [ ]:
from huggingface_hub import snapshot_download
model_dir = snapshot_download(MODEL_REPO, revision=MODEL_REVISION, allow_patterns=[
    "open_clip_config.json", "open_clip_model.safetensors", "tokenizer*", "special_tokens_map.json"])
model, preprocess = open_clip.create_model_from_pretrained("local-dir:" + model_dir)
tokenizer = open_clip.get_tokenizer("local-dir:" + model_dir)
model = model.to(DEVICE).eval()
USE_FP16 = DEVICE == "cuda"
print(preprocess)

In [ ]:
import csv
from torch.utils.data import Dataset, DataLoader

class Frames(Dataset):
    def __init__(self, items):
        self.items = items  # (video_id, row_index, image_number)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        video_id, row, n = self.items[i]
        path = BUNDLE / "images" / video_id / f"{n:03d}.jpg"
        if not path.is_file():
            return torch.zeros(3, 384, 384), i, False
        with Image.open(path) as image:
            return preprocess(image.convert("RGB")), i, True

items = []
for video_id in videos:
    with open(BUNDLE / "mapping" / f"{video_id}.csv", encoding="utf-8", newline="") as f:
        numbers = [int(r["n"]) for r in csv.DictReader(f)]
    if len(numbers) != manifest["videos"][video_id]["rows"]:
        raise SystemExit(f"{video_id}: mapping rows khác manifest")
    items += [(video_id, row, n) for row, n in enumerate(numbers)]

loader = DataLoader(Frames(items), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                    pin_memory=DEVICE == "cuda", persistent_workers=NUM_WORKERS > 0)
vectors = np.full((len(items), 768), np.nan, dtype=np.float16)
ok = np.zeros(len(items), dtype=bool)
started = time.time()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
    for step, (pixels, index, present) in enumerate(loader):
        feats = model.encode_image(pixels.to(DEVICE, non_blocking=True), normalize=True).float().cpu().numpy()
        keep = present.numpy()
        vectors[index.numpy()[keep]] = feats[keep].astype(np.float16)
        ok[index.numpy()[keep]] = True
        if step % 50 == 0:
            done = (step + 1) * BATCH_SIZE
            print(f"{min(done, len(items))}/{len(items)} | {done / (time.time() - started):.0f} img/s", flush=True)
seconds = time.time() - started
print(f"done {len(items)} in {seconds/60:.1f} min ({len(items)/seconds:.0f} img/s), missing {int((~ok).sum())}")

In [ ]:
# Sanity: text→image trên chính các vector vừa tính (phải ra ảnh hợp nghĩa).
probe = ["a news anchor in a studio", "a boat on a river", "people cooking in a kitchen"]
with torch.no_grad():
    t = model.encode_text(tokenizer(probe).to(DEVICE), normalize=True).float().cpu().numpy()
valid = np.where(ok)[0]
scores = vectors[valid].astype(np.float32) @ t.T
for j, q in enumerate(probe):
    top = valid[np.argsort(-scores[:, j])[:3]]
    print(q, "->", [(items[i][0], items[i][2], round(float(scores[np.searchsorted(valid, i), j]), 3)) for i in top])

In [ ]:
features = OUTPUT_DIR / "siglip2-features"
per_video, missing = {}, {}
offset = 0
for video_id in videos:
    rows = manifest["videos"][video_id]["rows"]
    block = vectors[offset:offset + rows]
    gaps = [int(items[offset + k][2]) for k in np.where(~ok[offset:offset + rows])[0]]
    offset += rows
    target = features / video_id.split("_")[0] / video_id / f"{video_id}.npy"
    target.parent.mkdir(parents=True, exist_ok=True)
    np.save(target, block)
    per_video[video_id] = {"rows": rows, "sha256": hashlib.sha256(target.read_bytes()).hexdigest()}
    if gaps:
        missing[video_id] = gaps
report = {"model_repo": MODEL_REPO, "model_revision": MODEL_REVISION, "open_clip": open_clip.__version__,
          "torch": torch.__version__, "device": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
          "dimension": 768, "dtype": "float16", "normalized": True, "metric": "IP",
          "preprocess": str(preprocess), "bundle_size": manifest.get("size"), "videos": len(videos),
          "rows": len(items), "missing_rows": int((~ok).sum()), "missing": missing,
          "seconds": round(seconds, 1), "per_video": per_video}
(features / "SIGLIP2_MANIFEST.json").write_text(json.dumps(report, indent=1))
archive = OUTPUT_DIR / "siglip2-features.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_STORED) as z:
    for path in sorted(features.rglob("*")):
        if path.is_file():
            z.write(path, path.relative_to(OUTPUT_DIR).as_posix())
print(archive, f"{archive.stat().st_size / 1e6:.0f} MB", "| missing rows:", report["missing_rows"])